In [ ]:
from matplotlib import pyplot as plt
from tqdm import tqdm
import itertools
import os
import numpy as np
import scipy.sparse
import pandas as pd
import re
import scanpy as sc
import random
import gc
import psutil

sc.set_figure_params(dpi=80)

In [ ]:
import math
from scipy.stats import linregress
import shutil
import scipy as sp

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'


In [ ]:
import importlib.util
import sys

def lazy_import(module_name, path_to_file):
    spec = importlib.util.spec_from_file_location(module_name,path_to_file)
    foo = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = foo
    spec.loader.exec_module(foo)
    return foo

utils_dir = "../../utils"
general_utils =  lazy_import("general_utils",os.path.join(utils_dir, "general_utils.py"))
preprocessing_utils = lazy_import("preprocessing_utils",os.path.join(utils_dir, "preprocessing_utils.py"))

from general_utils import ismember, grep, grep_exclude
from preprocessing_utils import kneepoint


In [ ]:
def create_diagonal_matrix(v, total_cells):
    cell_indexes = np.array(list(range(total_cells)))
    diagonal_matrix = scipy.sparse.csr_matrix((v,(cell_indexes, cell_indexes)), shape = [total_cells, total_cells])
    return(diagonal_matrix)

def get_average_neighborhood_expression(expression_matrix, spatial_neighborhood, anchor_cells = None, neighbor_cells = None):
    if anchor_cells is None:
        anchor_cells = np.array([True] * expression_matrix.shape[0])
    if neighbor_cells is None:
        neighbor_cells = np.array([True] * expression_matrix.shape[0])
    X_neighbors = expression_matrix[neighbor_cells,:].copy()
    spatial_neighborhood = spatial_neighborhood[anchor_cells,:].copy()[:,neighbor_cells].copy()
    num_neighbors_per_anchor = np.ravel(spatial_neighborhood.sum(axis=1))
    n_anchors = spatial_neighborhood.shape[0]
    normalization_matrix = create_diagonal_matrix(1/num_neighbors_per_anchor, n_anchors)
    average_expression_matrix = normalization_matrix @ spatial_neighborhood @ X_neighbors
    return(average_expression_matrix)

def get_neighborhood_fraction(spatial_neighborhood, boolean_variable):
    n_cells = spatial_neighborhood.shape[0]
    cell_indexes = np.array(list(range(n_cells)))
    masked_matrix = mask_neighborhood_columns(spatial_neighborhood, boolean_variable)
    boolean_fraction = np.ravel(masked_matrix.sum(axis=1)) / np.ravel(spatial_neighborhood.sum(axis=1))
    return(boolean_fraction)

def get_neighborhood_count(spatial_neighborhood, boolean_variable):
    n_cells = spatial_neighborhood.shape[0]
    cell_indexes = np.array(list(range(n_cells)))
    masked_matrix = mask_neighborhood_columns(spatial_neighborhood, boolean_variable)
    boolean_count = np.ravel(masked_matrix.sum(axis=1)) 
    total_count = np.ravel(spatial_neighborhood.sum(axis=1))
    return(boolean_count, total_count)

def mask_neighborhood_columns(spatial_neighborhood, mask):
    mask_matrix = create_diagonal_matrix(mask, spatial_neighborhood.shape[0])
    masked_neighborhood = spatial_neighborhood @ mask_matrix
    return(masked_neighborhood)

def mask_neighborhood_rows(spatial_neighborhood, mask):
    mask_matrix = create_diagonal_matrix(mask, spatial_neighborhood.shape[0])
    masked_neighborhood = mask_matrix @ spatial_neighborhood
    return(masked_neighborhood)

In [ ]:
tenx_data_dir = "../xenium_rawdata/"
output_path = '../xenium_preprocessing_outputs'

knn_output_path = os.path.join(output_path, 'knn')
pca_output_path = os.path.join(output_path, 'pca')
umap_output_path = os.path.join(output_path, 'umap')
adata_base_output_path = os.path.join(output_path, 'adata_base')
obs_output_path = os.path.join(output_path, 'obs')
updated_obs_output_path = os.path.join(output_path, 'obs_updated')
compiled_output_path = os.path.join(output_path, 'compiled')
spatialnhood_output_path = os.path.join(output_path, 'spatial_neighborhood')

os.makedirs(spatialnhood_output_path, exist_ok=True)

In [ ]:
def get_compiled_adata_subset(selected_tier, selected_subset, use_log, explained_var, umap_n_neighbors, cluster_n_neighbors, selected_layer = 'raw_in_nucleus', umap_min_dist = 0.05):
    dataset_prefix = "adata_" + selected_tier + "_SUBSET_%s_LAYER_%s_LOG_%s" % (selected_subset, selected_layer, use_log)
    file_prefix = "%s_EXPLAINEDVAR_%d_KNN_%s" % (dataset_prefix, int(explained_var * 100), "%d")
    
    adata_base_file = os.path.join(adata_base_output_path, dataset_prefix + ".h5ad")
    obs_file = os.path.join(obs_output_path,file_prefix % (cluster_n_neighbors) + "__obs.h5ad" )
    umap_file = os.path.join(umap_output_path,file_prefix % (umap_n_neighbors) + "_MINDIST_%d__umap.csv.gz" % int(umap_min_dist * 100))
    knn_file = os.path.join(knn_output_path,file_prefix % (cluster_n_neighbors) + "__knn.h5ad" )
    output_template = dataset_prefix + "_EXPLAINEDVAR_%d_CLUSTERKNN_%d_UMAPKNN_%d_MINDIST_%d__compiled.h5ad"
    output_file = os.path.join(compiled_output_path, output_template % (int(explained_var * 100), cluster_n_neighbors, umap_n_neighbors, int(umap_min_dist * 100)))

    print("Reading counts file")
    adata = sc.read_h5ad(adata_base_file)
    print("Reading obs file")
    obs = sc.read_h5ad(obs_file)
    print("Reading umap file")
    umap = pd.read_csv(umap_file, index_col = 0)
    
    adata.obsm['X_umap'] = umap.to_numpy()
    for colname in obs.obs.columns:
        adata.obs[colname] = pd.Categorical(obs.obs[colname])

    print("Loading master annotations")
    master_obs = sc.read_h5ad(os.path.join(updated_obs_output_path, "adata_tier0_SUBSET_all_LAYER_raw_in_nucleus_LOG_False_EXPLAINEDVAR_75_KNN_30__obs.h5ad"))
    master_obs = master_obs[adata.obs_names,:].copy()
    for my_obs in master_obs.obs.columns:
        adata.obs[my_obs] = master_obs.obs[my_obs].copy()
    
    return(adata, output_file)

In [ ]:
selected_tier = 'tier0'
selected_subset = 'all'
use_log = False
explained_var = 0.75
umap_n_neighbors = 10
cluster_n_neighbors = 30
selected_layer = 'raw_in_nucleus'
umap_min_dist = 0.1
adata, output_file = get_compiled_adata_subset(selected_tier = selected_tier, selected_subset = selected_subset, use_log = use_log, explained_var = explained_var, umap_n_neighbors = umap_n_neighbors, cluster_n_neighbors = cluster_n_neighbors, selected_layer = selected_layer, umap_min_dist = umap_min_dist)
dataset_prefix = "adata_" + selected_tier + "_SUBSET_%s" % (selected_subset)


In [ ]:
adata = sc.read_h5ad("../data/Reyes_Xenium_TIER_all_SUBSET_all__annotated.h5ad")

### Get spatial neighborhoods in entire dataset in single objects

In [ ]:
import spatialdata.models
from scipy import sparse
from spatialdata import to_polygons # New spatialdata
#from spatialdata._core.query._utils import circles_to_polygons
from shapely import Polygon, Point, affinity, intersection, MultiPolygon
from geopandas.geodataframe import GeoDataFrame

def get_rings(centers_x, centers_y, outer_radius, inner_radius):
    dilated_circles_outer = spatialdata.models.ShapesModel.parse(np.array([[x,y] for x,y in zip(centers_x, centers_y)]), geometry = 0, radius = outer_radius)
    dilated_circles_inner = spatialdata.models.ShapesModel.parse(np.array([[x,y] for x,y in zip(centers_x, centers_y)]), geometry = 0, radius = inner_radius)
    faces = to_polygons(dilated_circles_outer)
    holes = to_polygons(dilated_circles_inner)
    rings = GeoDataFrame(geometry = [Polygon(face.exterior.coords, holes = [hole.exterior.coords]) for face, hole in zip(faces['geometry'], holes['geometry'])])
    return(rings)

def get_circles(centers_x, centers_y, outer_radius):
    dilated_circles_outer = spatialdata.models.ShapesModel.parse(np.array([[x,y] for x,y in zip(centers_x, centers_y)]), geometry = 0, radius = outer_radius)
    circles = to_polygons(dilated_circles_outer)
    return(circles)

def get_spatial_polygon_neighborhood(adata, query_polygons):
    # Get all cell centroids into a GeoDataFrame
    all_cell_coordinates = [Point([x,y]) for x,y in zip(adata.obsm['X_spatial'][:,0], adata.obsm['X_spatial'][:,1])]
    all_cells = GeoDataFrame(geometry = all_cell_coordinates)
    
    # Spatial join operation
    joined = query_polygons.sjoin(all_cells)
    sources = joined.index.to_numpy()
    targets = joined['index_right'].to_numpy()
    df = pd.DataFrame({'sources': sources, 'targets': targets})
    
    return(df)

def get_query_polygons(adata, outer_radius = 60, inner_radius = None):
    # Pre-compute query_polygons for the entire dataset
    print("Pre-computing query polygons")
    centers_x = adata.obsm['X_spatial'][:,0]
    centers_y = adata.obsm['X_spatial'][:,1]
    circle_template = get_circles(centers_x, centers_y, outer_radius)
    query_polygons = circle_template
    return(query_polygons)

def compute_sample_neighborhoods(adata, slide_key, cell_type_key, selected_cell_types = [], outer_radius = 60, inner_radius = None):
    
    # Pre-compute cell_type_filter
    if(len(selected_cell_types) == 0):
        selected_cell_types = np.unique(adata.obs[cell_type_key])
    cell_type_filter = ismember(adata.obs[cell_type_key], selected_cell_types)[1]

    # Compute spatial neighborhoods per sample
    sources = np.array([], dtype=np.uint32)
    targets = np.array([], dtype=np.uint32)

    unique_slides = np.unique(adata.obs[slide_key])
    for my_slide in tqdm(unique_slides):
        slide_filter = adata.obs[slide_key] == my_slide
        selected_cell_indexes_slide = np.where(slide_filter)[0]
        adata_sub = adata[slide_filter,:].copy()
        
        selected_cells = np.logical_and(slide_filter, cell_type_filter)
        selected_cell_indexes_query = np.where(selected_cells)[0]
        query_polygons = get_query_polygons(adata[selected_cell_indexes_query,:], outer_radius = outer_radius, inner_radius = inner_radius)
        query_polygons.index = selected_cell_indexes_query.copy()
        
        result = get_spatial_polygon_neighborhood(adata_sub, query_polygons)
        
        sources = np.hstack((sources, result['sources'].to_numpy().astype(np.uint32)))
        targets = np.hstack((targets, selected_cell_indexes_slide[result['targets'].to_numpy().astype(np.uint32)]))
        
        print('RAM Used (GB):', psutil.virtual_memory()[3]/1000000000)
    spatial_neighborhood = scipy.sparse.csr_matrix(([1] * len(sources), (sources, targets)), shape = [adata.n_obs] * 2).astype(np.uint8)
    return(spatial_neighborhood)

def compute_distances_within_neighborhood(adata, obsm_field, neighbor_graph):
    """Computes euclidian distances within neighborhoods using arbitrary feature matrix

    Args:
        - adata: base adata object
        - obsm_field: feature matrix within the adata object (e.g. "X_spatial")
        - neighbor_graph: scipy.sparse.csr_matrix containing binary neighbor assignments. The number of rows is equal to adata.n_obs

    Returns
    -------
        - scipy.sparse.csr_matrix with distances between source and query cells, as specified in neighbor_graph.
    """
    assert (
        adata.n_obs == neighbor_graph.shape[0]
    ), "The number of cells in adata is different from number of cells in neighbor_graph"
    sources, targets = neighbor_graph.nonzero()
    source_vectors = adata.obsm[obsm_field][sources, :]
    target_vectors = adata.obsm[obsm_field][targets, :]
    dist = np.sqrt(np.power(source_vectors - target_vectors, 2).sum(axis=1))
    return sources, targets, dist

def filter_neighborhood(adata, spatial_neighborhood, min_radius = 0, max_radius = 60):
    sources, targets, dist = compute_distances_within_neighborhood(adata, "X_spatial", spatial_neighborhood)
    valid_entries = np.logical_and(dist >= min_radius, dist <= max_radius)
    sources = sources[valid_entries]
    targets = targets[valid_entries]
    dist = dist[valid_entries]
    result = scipy.sparse.csr_matrix(([1] * len(dist), (sources, targets)), shape=[adata.n_obs] * 2).astype(np.uint8)
    return(result)

/data1/lowes/reyesj3/miniconda3/envs/spatialdata/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/data1/lowes/reyesj3/miniconda3/envs/spatialdata/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [ ]:
# I choose a radius of 70um since my target radius is 60um.
# I noticed that since I work with polygons, the process of constructing a polygon that approximates a circle can exclude
# cells that are indeed within the radius. Thus, I overestimate the initial neighborhoods, and then refine them
# This could be improved using sklearn, but the polygon approach provides the possibility of estimating neighborhoods with
# more diverse geometries

outer_radius_expanded = 70
outer_radius = 60
inner_radius = None
spatial_neighbors = compute_sample_neighborhoods(
    adata, "slide_id", "cell_type_0", selected_cell_types = [], outer_radius = outer_radius_expanded, inner_radius = None
)
spatial_neighbors_updated = filter_neighborhood(adata, spatial_neighbors, min_radius = 0, max_radius = outer_radius)


adata_neighbors = sc.AnnData(scipy.sparse.csr_matrix(([], ([],[])), shape = [adata.n_obs, 1]))
adata_neighbors.obs_names = adata.obs_names
adata_neighbors.obsp['spatial_neighborhood'] = spatial_neighbors_updated.copy()

polygon_base = "circle"
output_file = "%s_MODE_%s_OUTER_%s_INNER_%s__spatialnhood.h5ad" % (dataset_prefix, polygon_base, str(outer_radius), str(inner_radius))
output_filename = os.path.join(spatialnhood_output_path, output_file)
sc.write(output_filename, adata=adata_neighbors)

  0%|          | 0/33 [00:00<?, ?it/s]

Pre-computing query polygons


  3%|▎         | 1/33 [00:20<11:08, 20.89s/it]

RAM Used (GB): 153.775357952
Pre-computing query polygons


  6%|▌         | 2/33 [00:45<11:46, 22.80s/it]

...

RAM Used (GB): 175.482564608
Pre-computing query polygons


 97%|█████████▋| 32/33 [20:30<00:21, 21.38s/it]

RAM Used (GB): 176.366833664
Pre-computing query polygons


100%|██████████| 33/33 [20:54<00:00, 38.01s/it]

RAM Used (GB): 176.629387264
